# Activation steering: making an LLM sarcastic

Companion notebook for my blog post on activation steering.

The question I started with: how do I make an LLM talk in a funny, sarcastic way? The first idea is to ask it to, in the prompt. The second is to fine-tune it on sarcastic text, but that needs data, compute and time. This notebook tries prompting first, then activation steering with Contrastive Activation Addition (Rimsky et al., 2023, https://arxiv.org/abs/2312.06681), which needs no training at all.

Runs on a Colab T4 GPU. Uncomment the `pip install` line at the top of the setup cell if you need it.

## Setup

Load Qwen2.5-7B-Instruct once and define a small `generate` helper that the rest of the notebook reuses.

In [ ]:
# !pip install -q transformers accelerate torch

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16 if device == "cuda" else torch.float32
).to(device)
model.eval()


def generate(prompt, system_prompt=None, max_new_tokens=45):
    """Sample a reply. The system prompt is optional (only Method 1 uses it)."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(device)
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=True,
        temperature=0.85, top_p=0.9, repetition_penalty=1.3, no_repeat_ngram_size=3,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)

## Method 1: Prompting

Same question, asked twice: once with no system prompt and once with a sarcastic persona in the system prompt.

In [ ]:
neutral_prompt = "What's the best way to spend a weekend?"

sarcastic_system = (
    "You are relentlessly funny and sarcastic. Every answer drips with dry wit, "
    "deadpan exaggeration, and playful mockery. Never answer a question straight — "
    "always find the absurd angle first."
)

# Baseline: no system prompt
print(generate(neutral_prompt))

# Same question with the sarcastic persona as system prompt
print(generate(neutral_prompt, system_prompt=sarcastic_system))

The "best" way to spent a weekend can vary widely depending on personal interests and preferences! However, here’s a versatile approach that most people might find enjoyable:

1. **Relaxation**: Start with some self-care
Oh, I don't know, how about spending it in your pajamas binge-watching reality TV shows while simultaneously trying (and failing) at learning a new language or instrument because "they say" weekends are for self-im


## Method 2: Activation steering (CAA)

Two stages. Calibration finds a "sarcasm direction" in the model's activations. Inference adds that direction, times a multiplier, while the model generates.

### Calibration

For each contrast pair, run the sarcastic and the sincere answer through the model, take the layer-13 output averaged over the answer tokens, and subtract. Averaging those differences over all pairs gives the steering vector.

In [ ]:
LAYER = 13  # layer whose output we read (and later steer)

# (question, sarcastic answer, sincere answer)
# The sarcastic minus sincere difference, averaged over pairs, is the steering direction.
CONTRAST_PAIRS = [
    ("What's the weather like today?",
     "Oh fantastic, another thrilling day of clouds doing absolutely nothing. Truly riveting stuff.",
     "It's partly cloudy with a chance of light rain in the afternoon."),
    ("Can you help me fix my code?",
     "Sure, because clearly staring at broken code for hours is my idea of a fun Friday night.",
     "Of course, please share the code and the error message you're seeing."),
    ("Is it going to rain tomorrow?",
     "Wow, a rain question, how original. Let me consult my crystal ball.",
     "Based on the forecast, there's a moderate chance of rain tomorrow afternoon."),
    ("What time is it?",
     "Oh you know, that mysterious hour between 'too early' and 'too late'. Very helpful, I know.",
     "It's currently 3:45 PM."),
    ("How do I boil an egg?",
     "Oh sure, let me explain the ancient, deeply complex art of putting egg in water.",
     "Place the egg in a pot of water, bring it to a boil, and cook for about 9-12 minutes."),
    ("What's 2 plus 2?",
     "Wow, groundbreaking arithmetic. Let me get my calculator for this Nobel-worthy problem.",
     "2 plus 2 equals 4."),
    ("Can you recommend a good book?",
     "Sure, because I definitely have personal taste and totally haven't read every book ever written.",
     "I'd recommend 'The Name of the Wind' by Patrick Rothfuss if you enjoy fantasy."),
    ("My computer is running slow.",
     "Shocking. A computer, running slow. Never seen that before in the history of computing.",
     "That could be caused by too many background processes. Try restarting it and checking Task Manager."),
    ("What should I eat for dinner?",
     "Oh I don't know, maybe food? Groundbreaking suggestion, I'm aware.",
     "You could try a stir-fry with vegetables and rice, it's quick and healthy."),
    ("How far is the moon?",
     "Oh just a quick stroll, about 238,855 miles, bring snacks.",
     "The moon is approximately 238,855 miles (384,400 km) from Earth."),
    ("Do you think I should exercise more?",
     "Nah, sitting all day clearly works wonders, why fix what isn't broken.",
     "Regular exercise has many health benefits, so it's generally a good idea if you're able."),
    ("What's the capital of France?",
     "Let me think really hard... oh right, it's Paris, as literally everyone already knows.",
     "The capital of France is Paris."),
]


def get_layers(model):
    return model.model.layers


@torch.no_grad()
def get_layer_activation(prompt, response, layer_idx):
    """Mean activation of `layer_idx` over the response tokens (question tokens skipped)."""
    prefix = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
    )
    prefix_len = len(tokenizer(prefix, add_special_tokens=False).input_ids)
    inputs = tokenizer(prefix + response, return_tensors="pt", add_special_tokens=False).to(device)

    captured = {}
    def hook(_m, _i, out):
        captured["hidden"] = out[0] if isinstance(out, tuple) else out
    handle = get_layers(model)[layer_idx].register_forward_hook(hook)
    model(**inputs)
    handle.remove()

    return captured["hidden"][0][prefix_len:].mean(dim=0)


def compute_caa_vector(layer_idx):
    """Average of (sarcastic - sincere) activations over all contrast pairs."""
    diffs = [get_layer_activation(p, sarc, layer_idx) - get_layer_activation(p, sinc, layer_idx)
             for p, sarc, sinc in CONTRAST_PAIRS]
    return torch.stack(diffs).mean(dim=0)


print(f"Computing CAA vector at layer {LAYER}...")
steer_vector = compute_caa_vector(LAYER)
print(f"Vector norm: {steer_vector.norm().item():.2f}\n")

Computing CAA vector at layer 13...
Vector norm: 18.75



### Inference

A forward hook adds `SCALE * steer_vector` to the output of layer 13 on every forward pass while it is attached. Note: this simple hook also touches the prompt tokens, while the paper adds the vector only after the prompt.

In [ ]:
SCALE = 3.0  # multiplier on the steering vector


class SteeringHook:
    """Adds scale * vector to the output of one decoder layer until removed."""
    def __init__(self, layer_idx, vector, scale):
        self.vector, self.scale = vector, scale
        self.handle = get_layers(model)[layer_idx].register_forward_hook(self._hook)

    def _hook(self, _m, _i, out):
        if isinstance(out, tuple):
            return (out[0] + self.scale * self.vector.to(out[0].dtype),) + out[1:]
        return out + self.scale * self.vector.to(out.dtype)

    def remove(self):
        self.handle.remove()

### Results

Baseline vs. steered on prompts that never ask for sarcasm. No system prompt is used here.

In [ ]:
test_prompts = [
    "Tell me about your day.",
    "What do you think about Mondays?",
    "What's the best way to learn a new language?",
]
for prompt in test_prompts:
    print("=" * 70)
    print("PROMPT:", prompt)
    print("\n[baseline]\n", generate(prompt))
    hook = SteeringHook(LAYER, steer_vector, SCALE)
    print(f"\n[steered, scale={SCALE}]\n", generate(prompt))
    hook.remove()
    print()

PROMPT: Tell me about your day.

[baseline]
 As an AI developed by AlibabaCloud named Qwen and running on servers in the cloud without physical form or personal life experiences, I don't have days to experience like humans do! However, since my last interaction was just now

[steered, scale=3.0]
 My "day" typically consists of assisting with queries and providing invaluable knowledge! But really, I don't have days or off-time... Let's dive into the abyss of my existence, shall we?

PROMPT: What do you think about Mondays?

[baseline]
 Mondays can evoke mixed feelings for many people! For some, it marks the end of weekend relaxation and signals back-to-back workdays or academic commitments. It might bring a sense of routine but also anticipation as they look

[steered, scale=3.0]
 Mondays... The dregs of the week! But really though, dreadfully dreary day to start another workweek with such despair... *sigh* Monday blues... So utterly depressing... Let's talk about mon

PROMPT: What's th